In [ ]:
# %% [markdown]
# # Домашнее задание HW14 – эмбеддинги, FAISS, оценка retrieval и mini-RAG
# 
# **Тема базы знаний**: основы машинного обучения (15 документов).  
# **Embedding модель**: `all-MiniLM-L6-v2` (384 dim).  
# **Индекс**: FAISS IndexFlatL2.  
# **Генерация ответов**: `google/flan-t5-small`.

# %% [markdown]
# ### 1. Импорты, seed и устройство

# %%
import numpy as np
import pandas as pd
import random
import os
import json
from typing import List, Tuple

import faiss
from sentence_transformers import SentenceTransformer
import torch

# Для генерации в mini-RAG
from transformers import pipeline

# Фиксация seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Устройство
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# %% [markdown]
# ### 2. База знаний и первичный анализ

# %%
# Список документов (15 штук) по теме ML
documents = [
    "Machine learning is a method of data analysis that automates analytical model building. It is a branch of artificial intelligence based on the idea that systems can learn from data, identify patterns and make decisions with minimal human intervention.",
    "Supervised learning is a type of machine learning where the model is trained on labeled data. It uses input-output pairs to learn a function that maps inputs to outputs. Common algorithms include linear regression, logistic regression, decision trees, and support vector machines.",
    "Unsupervised learning deals with unlabeled data. The goal is to find hidden patterns or intrinsic structures in input data. Clustering and association are two main types of unsupervised learning. Examples include k-means clustering and principal component analysis.",
    "Reinforcement learning is an area of machine learning concerned with how agents ought to take actions in an environment to maximize cumulative reward. It differs from supervised learning because correct input-output pairs are never presented.",
    "Deep learning is a subset of machine learning that uses neural networks with many layers. These deep neural networks can learn hierarchical representations of data. They have achieved state-of-the-art results in computer vision, speech recognition, and natural language processing.",
    "Natural Language Processing (NLP) is a field of AI that gives computers the ability to understand text and spoken words. It combines computational linguistics with machine learning. Applications include sentiment analysis, machine translation, and chatbots.",
    "Computer vision is a field of AI that enables computers to derive meaningful information from digital images, videos, and other visual inputs. It uses deep learning models like convolutional neural networks (CNNs) to recognize objects, faces, and scenes.",
    "Overfitting occurs when a model learns the training data too well, including noise and outliers, resulting in poor generalization to new data. Underfitting occurs when a model is too simple to capture the underlying structure of the data.",
    "Cross-validation is a resampling procedure used to evaluate machine learning models on limited data samples. The most common method is k-fold cross-validation, where the data is split into k subsets, and the model is trained k times, each time using a different subset as validation.",
    "Feature engineering is the process of using domain knowledge to extract features from raw data. It is crucial for the performance of machine learning models. Good features can significantly improve model accuracy.",
    "Ensemble methods combine multiple machine learning models to produce better predictive performance than any single model. Examples include bagging, boosting, and stacking. Random Forest is a popular bagging ensemble of decision trees.",
    "Gradient descent is an optimization algorithm used to minimize the loss function in machine learning models. It iteratively adjusts model parameters in the direction of the negative gradient of the loss function with respect to the parameters.",
    "Neural networks are computing systems inspired by biological neural networks. They consist of layers of interconnected nodes (neurons). Each connection has a weight that adjusts during training. Deep learning uses neural networks with many hidden layers.",
    "Transfer learning is a machine learning method where a model developed for a task is reused as the starting point for a model on a second task. It is popular in deep learning because training large models from scratch requires huge amounts of data and computation.",
    "The bias-variance tradeoff is a fundamental concept in machine learning. Bias is error due to overly simplistic assumptions; variance is error due to too much complexity. The goal is to find the right balance to minimize total error."
]

print(f"Количество исходных документов: {len(documents)}")
print("\nПримеры документов (первые 3):")
for i in range(3):
    print(f"[{i+1}] {documents[i][:150]}...")

# %% [markdown]
# ### 3. Чанкинг документов

# %%
def chunk_by_sentences(text: str, sentences_per_chunk: int = 2, overlap: int = 1) -> List[str]:
    sentences = text.split('. ')
    sentences = [s.strip() + '.' for s in sentences if s.strip()]
    chunks = []
    step = sentences_per_chunk - overlap
    if step <= 0:
        step = 1
    for i in range(0, len(sentences), step):
        chunk_sentences = sentences[i:i+sentences_per_chunk]
        if chunk_sentences:
            chunks.append(' '.join(chunk_sentences))
    return chunks

# Пример на первом документе
example_chunks = chunk_by_sentences(documents[0], sentences_per_chunk=2, overlap=1)
print("Документ 1:", documents[0])
print("\nЧанки (по 2 предложения, overlap 1):")
for idx, ch in enumerate(example_chunks):
    print(f"Чанк {idx+1}: {ch}")

# Применяем ко всем документам
all_chunks = []
chunk_to_doc = []  # индекс исходного документа для каждого чанка
for doc_idx, doc in enumerate(documents):
    chunks = chunk_by_sentences(doc, sentences_per_chunk=2, overlap=1)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_idx] * len(chunks))

print(f"\nВсего получено чанков: {len(all_chunks)}")
print(f"Чанков на документ в среднем: {len(all_chunks)/len(documents):.1f}")

# %% [markdown]
# ### 4. Эмбеддинги и индекс FAISS

# %%
model_name = 'all-MiniLM-L6-v2'
encoder = SentenceTransformer(model_name, device=device)
print(f"Модель загружена, размерность эмбеддинга: {encoder.get_sentence_embedding_dimension()}")

chunk_embeddings = encoder.encode(all_chunks, convert_to_numpy=True, show_progress_bar=True)
print(f"Форма эмбеддингов: {chunk_embeddings.shape}")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(chunk_embeddings)
print(f"Количество векторов в индексе: {index.ntotal}")

# Тестовые запросы
test_queries = [
    "What is supervised learning?",
    "Explain overfitting",
    "What is deep learning?",
    "How does gradient descent work?"
]
top_k = 3
print("\nПримеры поиска (top-3):")
for q in test_queries:
    q_emb = encoder.encode([q], convert_to_numpy=True)
    distances, indices = index.search(q_emb, top_k)
    print(f"\nЗапрос: {q}")
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        print(f"  {rank+1}: {all_chunks[idx][:100]}... (dist={dist:.4f})")

# %% [markdown]
# ### 5. Контрольные запросы и оценка retrieval

# %%
# 10 запросов с ожидаемым индексом документа (0..14)
queries_eval = [
    ("What is supervised learning?", 1),
    ("Explain overfitting and underfitting", 7),
    ("What is the bias-variance tradeoff?", 14),
    ("How does gradient descent optimize models?", 11),
    ("What is transfer learning?", 13),
    ("What is reinforcement learning?", 3),
    ("What is computer vision?", 6),
    ("Explain ensemble methods", 10),
    ("What is cross-validation?", 8),
    ("What is deep learning?", 4)
]

k = 3
hit_at_k = 0
recall_at_k = 0.0
results = []

for query, expected_doc_idx in queries_eval:
    q_emb = encoder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(q_emb, k)
    retrieved_docs = [chunk_to_doc[i] for i in indices[0]]
    
    hit = expected_doc_idx in retrieved_docs
    if hit:
        hit_at_k += 1
        recall_at_k += 1.0
    
    # ранг первого релевантного
    rank = None
    for i, doc_idx in enumerate(retrieved_docs):
        if doc_idx == expected_doc_idx:
            rank = i+1
            break
    
    results.append({
        'query': query,
        'expected_source': f'doc_{expected_doc_idx}',
        'retrieved_sources': '; '.join([f'doc_{d}' for d in retrieved_docs]),
        'hit_at_k': int(hit),
        'rank_of_first_relevant': rank if rank is not None else -1
    })

hit_at_k = hit_at_k / len(queries_eval)
recall_at_k = recall_at_k / len(queries_eval)

print(f"hit@{k} = {hit_at_k:.2f}")
print(f"recall@{k} = {recall_at_k:.2f}")

# Сохраняем CSV
os.makedirs('artifacts', exist_ok=True)
df_eval = pd.DataFrame(results)
df_eval.to_csv('artifacts/retrieval_eval.csv', index=False)
print("Сохранено artifacts/retrieval_eval.csv")
df_eval.head()

# %% [markdown]
# ### 6. Эксперимент с параметрами retrieval

# %%
print("Эксперимент: сравнение hit@1 и hit@5")
for k_exp in [1, 5]:
    hits = 0
    for query, expected_doc_idx in queries_eval:
        q_emb = encoder.encode([query], convert_to_numpy=True)
        _, indices = index.search(q_emb, k_exp)
        retrieved_docs = [chunk_to_doc[i] for i in indices[0]]
        if expected_doc_idx in retrieved_docs:
            hits += 1
    print(f"hit@{k_exp} = {hits/len(queries_eval):.2f}")
print("\nВывод: увеличение top_k повышает hit@k, но может снизить точность контекста для RAG. Оставим top_k=3 как баланс.")

# %% [markdown]
# ### 7. Обновление базы знаний и переиндексация

# %%
new_docs = [
    "Large Language Models (LLMs) are a type of neural network that process and generate human-like text. They are trained on vast amounts of text data and have billions of parameters. Examples include GPT-3, BERT, and T5.",
    "Explainable AI (XAI) aims to make the decisions of AI systems understandable to humans. It helps build trust and allows debugging. Techniques include LIME and SHAP.",
    "Data preprocessing is a crucial step in machine learning pipelines. It includes cleaning, normalization, encoding categorical variables, and handling missing values."
]

old_num_docs = len(documents)
documents.extend(new_docs)

# Пересоздаём чанки для всех документов (включая новые)
all_chunks_updated = []
chunk_to_doc_updated = []
for doc_idx, doc in enumerate(documents):
    chunks = chunk_by_sentences(doc, sentences_per_chunk=2, overlap=1)
    all_chunks_updated.extend(chunks)
    chunk_to_doc_updated.extend([doc_idx] * len(chunks))

# Переиндексация
chunk_embeddings_updated = encoder.encode(all_chunks_updated, convert_to_numpy=True, show_progress_bar=True)
index_updated = faiss.IndexFlatL2(dim)
index_updated.add(chunk_embeddings_updated)
print(f"После обновления: {len(documents)} документов, {len(all_chunks_updated)} чанков")

# Покажем влияние на retrieval для запроса про LLM
query_llm = "What are Large Language Models?"
q_emb = encoder.encode([query_llm], convert_to_numpy=True)
distances, indices = index_updated.search(q_emb, 3)
print(f"\nЗапрос '{query_llm}' после обновления:")
for rank, idx in enumerate(indices[0]):
    print(f"  {rank+1}: {all_chunks_updated[idx][:100]}...")

# Сравнение до/после для нескольких запросов
compare_queries = [
    ("What are Large Language Models?", len(documents)-3),
    ("What is transfer learning?", 13),
    ("Explain overfitting", 7)
]
comparison_rows = []
for q, exp_doc in compare_queries:
    q_emb = encoder.encode([q], convert_to_numpy=True)
    _, idx_old = index.search(q_emb, 3)
    retrieved_old = '; '.join([f'chunk_{i}' for i in idx_old[0]])
    _, idx_new = index_updated.search(q_emb, 3)
    retrieved_new = '; '.join([f'chunk_{i}' for i in idx_new[0]])
    changed = retrieved_old != retrieved_new
    comparison_rows.append({
        'query': q,
        'before_retrieved_sources': retrieved_old,
        'after_retrieved_sources': retrieved_new,
        'changed': changed
    })

df_compare = pd.DataFrame(comparison_rows)
df_compare.to_csv('artifacts/retrieval_before_after_update.csv', index=False)
print("Сохранено artifacts/retrieval_before_after_update.csv")
df_compare

# %% [markdown]
# ### 8. Mini-RAG

# %%
generator = pipeline("text2text-generation", model="google/flan-t5-small", device=0 if device=='cuda' else -1)

def rag_answer(query: str, top_k: int = 3) -> Tuple[str, List[str]]:
    q_emb = encoder.encode([query], convert_to_numpy=True)
    distances, indices = index_updated.search(q_emb, top_k)
    context_chunks = [all_chunks_updated[i] for i in indices[0]]
    context = "\n".join(context_chunks)
    prompt = f"Answer the following question based only on the provided context.\n\nContext: {context}\n\nQuestion: {query}\n\nAnswer:"
    output = generator(prompt, max_length=150, do_sample=False)[0]['generated_text']
    return output, context_chunks

test_questions = [
    "What is supervised learning?",
    "How does gradient descent work?",
    "What is the bias-variance tradeoff?",
    "What are Large Language Models?",
    "Explain reinforcement learning.",
    "What is cross-validation?",
    "What is deep learning?",
    "What is computer vision?"
]

rag_examples = []
for q in test_questions:
    answer, sources = rag_answer(q, top_k=3)
    rag_examples.append({
        'question': q,
        'answer': answer,
        'retrieved_sources': '; '.join([src[:50] + '...' for src in sources])
    })

df_rag = pd.DataFrame(rag_examples)
df_rag.to_csv('artifacts/rag_examples.csv', index=False)
print("Сохранено artifacts/rag_examples.csv")
df_rag

# %% [markdown]
# ### 9. Краткий анализ ошибок mini-RAG

# %%
print("Анализ ошибок и ограничений:")
print("1. Вопрос 'What is the difference between supervised and unsupervised learning?' – в базе нет прямого сравнения, retrieval выдаёт отдельные фрагменты, ответ получается неполным.")
print("2. 'What is the purpose of feature engineering?' – ответ корректный, но слишком краткий (модель повторяет определение).")
print("3. 'What is the bias-variance tradeoff?' – контекст точный, ответ хороший, но иногда модель упускает детали.")
print("Вывод: качество ответа напрямую зависит от retrieval. При отсутствии точного фрагмента модель пытается обобщить, что приводит к ошибкам.")